# 3.3 Lokalisierung & Regionales Mapping

Dieses Notebook lädt ein bereits trainiertes SOM-Modell sowie die zugrundeliegenden Daten und ordnet neue Datenpunkte (gefiltert nach Koordinaten) den bestehenden Hexbins zu.
Es generiert einen Bericht, der die globalen Karten (TDS, Temperatur, Gestein) zeigt, aber zusätzlich die Anzahl der Treffer (**n**) aus der gewählten Region in jedem Hexbin anzeigt.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import pickle
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LogNorm
from minisom import MiniSom

sns.set_theme(style="whitegrid")
print("Libraries geladen.")

Libraries geladen.


In [2]:
# --------- deutscher kommentar ---------
# Konfiguration der Region (Standard: Oberrheingraben)
# ---------------------------------------
LAT_MIN = float(os.environ.get('LOC_LAT_MIN', 47.30))
LAT_MAX = float(os.environ.get('LOC_LAT_MAX', 50.28))
LON_MIN = float(os.environ.get('LOC_LON_MIN', 7.45))
LON_MAX = float(os.environ.get('LOC_LON_MAX', 9.60))
REGION_NAME = os.environ.get('LOC_REGION_NAME', 'Oberrheingraben')

# Suche das Wurzelverzeichnis (abschlussarbeit)
base_dir = Path.cwd().resolve()
while base_dir.name != "abschlussarbeit" and base_dir.parent != base_dir:
    base_dir = base_dir.parent
results_dir = base_dir / "Abschlussarbeit Bearbeitung" / "Jupyter Notebooks" / "3_Machine-Learning" / "3.2_Machine-Learning" / "MiniSom" / "MiniSom_Results"

print(f"Suche Modelle in: {results_dir}")
print(f"Filterregion: {REGION_NAME} (Lat {LAT_MIN}-{LAT_MAX}, Lon {LON_MIN}-{LON_MAX})")

Suche Modelle in: C:\Users\lucca\Desktop\Abschlussarbeit HTWK\abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results
Filterregion: Oberrheingraben (Lat 47.3-50.28, Lon 7.45-9.6)


In [3]:
# --------- deutscher kommentar ---------
# Hilfsfunktionen für das Plotting (ähnlich wie im Hauptskript)
# ---------------------------------------
def calculate_hex_map(values, weights, som_x, som_y):
    # Hilfsfunktion um Mittelwerte pro Hex-Bin zu berechnen
    res = np.full((som_y, som_x), np.nan)
    for y in range(som_y):
        for x in range(som_x):
            mask = (weights[:, 0] == x) & (weights[:, 1] == y)
            if np.any(mask):
                res[y, x] = np.mean(values[mask])
    return res

def plot_background_with_n(ax, background_matrix, counts, som_x, som_y, title, cmap='viridis', norm=None, is_categorical=False, rock_color_map=None):
    ax.set_aspect('equal')
    
    for y in range(som_y):
        for x in range(som_x):
            offset = 0.5 if y % 2 != 0 else 0.0
            center_x = x + offset
            center_y = y * (np.sqrt(3) / 2)
            
            val = background_matrix[y, x]
            count = int(counts[y, x])
            
            # Hintergrundfarbe
            fc = 'lightgrey'
            if is_categorical:
                if val is not None and rock_color_map and str(val) in rock_color_map:
                    fc = rock_color_map[str(val)]
            else:
                if not pd.isna(val):
                    if norm:
                        fc = plt.get_cmap(cmap)(norm(val))
                    else:
                        # Normalize manually if no norm given
                        fc = plt.get_cmap(cmap)(0.5)

            hex_poly = mpatches.RegularPolygon((center_x, center_y), numVertices=6, 
                                               radius=1/np.sqrt(3)*0.95, orientation=np.radians(30),
                                               facecolor=fc, edgecolor='k', alpha=0.8)
            ax.add_patch(hex_poly)
            
            # Overlay Text
            # Zeige [Bin-Indizes] und n-Anzahl
            txt_color = 'white' if not is_categorical and not pd.isna(val) and plt.get_cmap(cmap)(norm(val) if norm else 0.5)[0] < 0.5 else 'black'
            ax.text(center_x, center_y+0.1, f"[{x+1},{y+1}]", ha='center', fontsize=6, color=txt_color, fontweight='bold')
            
            if count > 0:
                ax.text(center_x, center_y-0.15, f"n={count}", ha='center', va='center', 
                        fontsize=10, fontweight='black', color='red', 
                        path_effects=[]) # Hier könnte man noch path_effects für bessere Lesbarkeit adden

    ax.set_xlim(-1, som_x + 0.5)
    ax.set_ylim(-0.5, som_y * (np.sqrt(3)/2) + 0.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12)
    return ax

In [4]:
# --------- deutscher kommentar ---------
# Finde Daten und aktuellsten Lauf
# ---------------------------------------
# Suche das Wurzelverzeichnis (abschlussarbeit)
base_dir = Path.cwd().resolve()
while base_dir.name != "abschlussarbeit" and base_dir.parent != base_dir:
    base_dir = base_dir.parent

results_dir = base_dir / "Abschlussarbeit Bearbeitung" / "Jupyter Notebooks" / "3_Machine-Learning" / "3.2_Machine-Learning" / "MiniSom" / "MiniSom_Results"
subdirs = [d for d in results_dir.iterdir() if d.is_dir()]
latest_run_dir = max(subdirs, key=os.path.getmtime)

prep_dir = base_dir / "Abschlussarbeit Bearbeitung" / "Jupyter Notebooks" / "3_Machine-Learning" / "3.1_Preprocessing" / "Preprocessing"
prep_subdirs = [d for d in prep_dir.iterdir() if d.is_dir()]
latest_prep = max(prep_subdirs, key=os.path.getmtime)
csv_path = latest_prep / "Preprocessed_SOM_Ready.csv"

df_full = pd.read_csv(csv_path)
print(f"Setup abgeschlossen. Exportiere n-Berichte für {REGION_NAME}...")


Setup abgeschlossen. Exportiere n-Berichte für Oberrheingraben...


In [5]:
model_files = list(latest_run_dir.glob("MODEL_*.pkl"))

for model_file in model_files:
    run_id = model_file.stem.replace("MODEL_", "")
    with open(model_file, 'rb') as f: m_data = pickle.load(f)
    som, scaler, train_cols = m_data['som'], m_data['scaler'], m_data['train_cols']
    som_x, som_y = som.get_weights().shape[1], som.get_weights().shape[0]
    
    # 1. BMUs für ALLE Daten berechnen (für Hintergrund-Map)
    df_all_valid = df_full.dropna(subset=train_cols).copy()
    all_scaled = scaler.transform(df_all_valid[train_cols].values)
    all_bmus = np.array([som.winner(x) for x in all_scaled])
    df_all_valid['som_x'], df_all_valid['som_y'] = all_bmus[:, 0], all_bmus[:, 1]

    # 2. Filter für die REGION
    df_region = df_all_valid[
        (df_all_valid['WGS84_latitude'] >= LAT_MIN) & (df_all_valid['WGS84_latitude'] <= LAT_MAX) &
        (df_all_valid['WGS84_longitude'] >= LON_MIN) & (df_all_valid['WGS84_longitude'] <= LON_MAX)
    ].copy()
    
    if df_region.empty:
        print(f"  [SKIP] Keine Daten für Region in Run {run_id}")
        continue
    
    # Counts pro Zelle für die Region
    counts = np.zeros((som_y, som_x))
    region_bmus = df_region[['som_x', 'som_y']].values.astype(int)
    for bx, by in region_bmus: counts[by, bx] += 1

    # 3. Hintergrund-Matrizen (Durchschnittswerte)
    tds_mask = df_all_valid['total_dissolved_solids_in_mmol/L'].notna()
    tds_bg = calculate_hex_map(df_all_valid.loc[tds_mask, 'total_dissolved_solids_in_mmol/L'].values, all_bmus[tds_mask], som_x, som_y)
    
    temp_mask = df_all_valid['temperature_in_c'].notna()
    temp_bg = calculate_hex_map(df_all_valid.loc[temp_mask, 'temperature_in_c'].values, all_bmus[temp_mask], som_x, som_y)
    
    # ROCK TYPE Background
    dom_rocks = df_all_valid.groupby(['som_x', 'som_y'])['rock_type'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else None).to_dict()
    unique_rocks = sorted(df_all_valid['rock_type'].dropna().unique())
    pal = sns.color_palette('Set3', n_colors=len(unique_rocks))
    rock_color_map = {str(rt): pal[i] for i, rt in enumerate(unique_rocks)}
    rock_bg = np.full((som_y, som_x), None, dtype=object)
    for y in range(som_y):
        for x in range(som_x):
            rock_bg[y, x] = dom_rocks.get((x, y), None)

    # PDF Export
    pdf_path = latest_run_dir / f"LOCALIZATION_{REGION_NAME}_{run_id}.pdf"
    with PdfPages(pdf_path) as pdf:
        # Seite 1: TDS
        f, ax = plt.subplots(figsize=(8,8))
        norm = LogNorm(vmin=df_all_valid['total_dissolved_solids_in_mmol/L'].min(), vmax=df_all_valid['total_dissolved_solids_in_mmol/L'].max())
        plot_background_with_n(ax, tds_bg, counts, som_x, som_y, f"Region: {REGION_NAME}\nHintergrund: TDS [mmol/L] (All Data)", norm=norm)
        pdf.savefig(f); plt.close(f)
        
        # Seite 2: Temperatur
        f, ax = plt.subplots(figsize=(8,8))
        t_vals = df_all_valid['temperature_in_c'].dropna()
        norm_t = plt.Normalize(vmin=t_vals.min(), vmax=t_vals.max()) if not t_vals.empty else None
        plot_background_with_n(ax, temp_bg, counts, som_x, som_y, f"Region: {REGION_NAME}\nHintergrund: Temperatur [°C] (All Data)", cmap='coolwarm', norm=norm_t)
        pdf.savefig(f); plt.close(f)
        
        # Seite 3: Gestein
        f, ax = plt.subplots(figsize=(8,8))
        plot_background_with_n(ax, rock_bg, counts, som_x, som_y, f"Region: {REGION_NAME}\nHintergrund: Dominante Gesteinsart (All Data)", is_categorical=True, rock_color_map=rock_color_map)
        # Legende für Gesteine
        legend_handles = [mpatches.Patch(color=rock_color_map[str(r)], label=str(r)) for r in unique_rocks]
        ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, fontsize=8)
        pdf.savefig(f); plt.close(f)

    print(f"  -> {pdf_path.name} erstellt.")


  -> LOCALIZATION_Oberrheingraben_004_Plus_pH-K-Fe_Na-Mg-Ca-Cl-SO4-HCO3-pH-K-Fe.pdf erstellt.
